#Important Installs

In [ ]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=93611fef43c1d1d1d886bf58b37250520f38afa1da8d7524d80fd5f63c69f7b0
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


#Imports

In [ ]:
import tarfile
import os
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import torch
import torch.nn as nn
from torch.optim import AdamW, Adam
from datasets import load_dataset
from transformers import BartForConditionalGeneration, BartTokenizer

import evaluate
from tqdm import tqdm
import nltk
nltk.download('wordnet')
nltk.download('punkt')


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

# Universal Variable

For reproducability

## Setting up Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Set device to cuda if available

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Setting Random Seed

In [ ]:
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)

## Links/ Paths

In [ ]:
zippedDataSetPath = '/content/drive/MyDrive/Natural_Language_Processing/Dataset/english_XLSum_v2.0.tar.bz2'
extractedDataSetPath = '/content/xlsum_en_dataset'

modelSavePath = '/content/drive/MyDrive/Natural_Language_Processing/Model_Saves'

## Hyperparameters

In [ ]:
MODEL_RUN_FILENAME = 'Bart-Summarisation-Final-3.pth'

MAX_SUMMARY_LENGTH = 64
MAX_INPUT_LENGTH = 512

BEAM_SEARCH_WIDTH = 4

BATCH_SIZE = 256
EPOCHS = 1

LEARNING_RATE = 1e-5

# Extracting and Loading Dataset

In [ ]:
# Create Directory if it does not exist
os.makedirs(extractedDataSetPath, exist_ok=True)

# Extract the tar file
with tarfile.open(zippedDataSetPath, 'r:bz2') as tar:
    tar.extractall(path=extractedDataSetPath)

print("Files have been successfully extracted.")

/tmp/ipykernel_404/1810096438.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extractedDataSetPath)


Files have been successfully extracted.


In [ ]:
xlsumEnDataset = load_dataset('json', data_files={
    'test': os.path.join(extractedDataSetPath, 'english_test.jsonl'),
    'val': os.path.join(extractedDataSetPath, 'english_val.jsonl')
})

Generating test split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

In [ ]:
print(xlsumEnDataset['test'][0]['title'])
print(xlsumEnDataset['test'][0]['summary'])
print(xlsumEnDataset['test'][0]['text'])

Trump inauguration speech: 'Angry', 'authentic', 'primal'
Donald Trump campaigned on becoming a president unlike any Washington has ever seen. With his inauguration speech, he's already set the tone.
By Kate DaileyBBC News Earlier this week, Trump posted a photo of himself sitting at a desk at Mar-a-Largo, a permanent marker hovering over a notepad. "Writing my inaugural address at the Winter White House, Mar-a-Lago, three weeks ago. Looking forward to Friday," he tweeted. Trump vows to end 'American carnage' Trump's angry call to arms Full text of Trump's inauguration speech It's unclear whether the president-elect actually wrote the speech himself, but the content was pure Trump: the same populist message that resonated throughout the primaries and the campaign. "Today, we are not merely transferring power from one administration to another, or from one party to another, but we are transferring power from Washington, DC, and giving it back to you, the people," he said at the beginnin

# Initializing tokenizer and Model

In [ ]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

# Since we are generating text for summerization, we shall be using the BartForConditionalGenration specifically
model1 = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
model2 = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
model3 = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

## Defining tokenize function for dataset tokenization

In [ ]:
def tokenize(batch):
  # We shall truncate the input text to Bart's max length, which is 1024
  modelInput = tokenizer(batch['text'],padding='max_length',truncation=True,max_length=MAX_INPUT_LENGTH)
  # And truncate the summary to a length of 128
  labels = tokenizer(batch['summary'],padding='max_length',truncation=True,max_length=MAX_SUMMARY_LENGTH)

  modelInput['labels'] = labels['input_ids']
  return modelInput


## Actual mapping of tokenize function

In [ ]:
testDs = xlsumEnDataset['test'].map(tokenize,batched=True)
valDs = xlsumEnDataset['val'].map(tokenize,batched=True)

Map:   0%|          | 0/11535 [00:00<?, ? examples/s]

Map:   0%|          | 0/11535 [00:00<?, ? examples/s]

In [ ]:
testDs.set_format('torch',columns=['input_ids','attention_mask','labels'])
valDs.set_format('torch',columns=['input_ids','attention_mask','labels'])

# Evaluate Model Function

In [ ]:
def evaluate_model(model,testLoader,tokenizer, device):
  model.eval()
  rougeMetric = evaluate.load('rouge')
  bleuMetric = evaluate.load('bleu')
  meteorMetric = evaluate.load('meteor')

  generatedSummary = []
  originalSummary = []

  # setting torch to not use gradients
  with torch.no_grad():
    for batch in tqdm(testLoader, desc="Evaluating"):
      inputId = batch['input_ids'].to(device)
      attentionMask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      # replace errors for padding
      labels = torch.where(labels != -100, labels, tokenizer.pad_token_id)

      # when generating summary, we set the max length of summary the same as initilised and bean search width to be 4 and we set early stopping to be true incase there is no improvement
      generatedId = model.generate(input_ids = inputId, attention_mask = attentionMask, max_length=MAX_SUMMARY_LENGTH, num_beams=BEAM_SEARCH_WIDTH, early_stopping=True)

      # decoding the original and generated inputIds
      decodedGeneratedSummary = tokenizer.batch_decode(generatedId,skip_special_tokens=True)
      decodedOriginalSummary = tokenizer.batch_decode(labels,skip_special_tokens=True)

      # Filter out any None values or empty strings to prevent errors in the evaluate library
      decodedGeneratedSummary = [s for s in decodedGeneratedSummary if s is not None and s.strip() != '']
      decodedOriginalSummary = [s for s in decodedOriginalSummary if s is not None and s.strip() != '']

      generatedSummary.extend(decodedGeneratedSummary)
      originalSummary.extend(decodedOriginalSummary)


  # Computing rouge scores, while considering words with the same stem to be equal
  rougeResult = rougeMetric.compute(predictions=generatedSummary, references=originalSummary, use_stemmer=True)
  bleuResult = bleuMetric.compute(predictions=generatedSummary, references=originalSummary)
  meteorResult = meteorMetric.compute(predictions=generatedSummary, references=originalSummary)

  # Combine all results
  fullResult = {}
  fullResult.update(rougeResult)
  fullResult.update(bleuResult)
  fullResult.update(meteorResult)

  return fullResult

# Loading model, weights and dataset

In [ ]:
torch.cuda.empty_cache()

## Loading Models and weights

---
Here Model 1 is the model with the following parameters:

MAX_SUMMARY_LENGTH = 128
MAX_INPUT_LENGTH = 1024
BEAM_SEARCH_WIDTH = 4

---

Model 2 (around 10000 parameters) and Model 3 (the entire dataset) are with the following parameters:

MAX_SUMMARY_LENGTH = 64
MAX_INPUT_LENGTH = 512
BEAM_SEARCH_WIDTH = 4

In [ ]:
model1.load_state_dict(torch.load(os.path.join(modelSavePath,'Bart-Summarisation-Final-1.pth'), weights_only=True))
model2.load_state_dict(torch.load(os.path.join(modelSavePath,'Bart-Summarisation-Final-2-test.pth'), weights_only=True))
model3.load_state_dict(torch.load(os.path.join(modelSavePath,'Bart-Summarisation-Final-3.pth'), weights_only=True))

<All keys matched successfully>

## Loading Dataset

In [ ]:
testLoader = torch.utils.data.DataLoader(testDs, batch_size=BATCH_SIZE, shuffle=False)
valLoader = torch.utils.data.DataLoader(valDs, batch_size=BATCH_SIZE, shuffle=False)

# Evaluating Models

In [ ]:
model1.to(device)

performanceMetrics1 = None
performanceMetrics1 = evaluate_model(model1,testLoader,tokenizer, device)

del model1

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Evaluating: 100%|██████████| 46/46 [01:28<00:00,  1.93s/it]


IndexError: list index out of range

In [ ]:
model2.to(device)

performanceMetrics2 = evaluate_model(model2,testLoader,tokenizer, device)

del model2

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Evaluating: 100%|██████████| 46/46 [12:17<00:00, 16.02s/it]


In [ ]:
model3.to(device)

performanceMetrics3 = evaluate_model(model3,testLoader,tokenizer, device)

del model3

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Evaluating: 100%|██████████| 46/46 [07:20<00:00,  9.58s/it]


In [ ]:
print(performanceMetrics1)
print(performanceMetrics2)
print(performanceMetrics3)

None
{'rouge1': np.float64(0.2028865547939851), 'rouge2': np.float64(0.027796086307394614), 'rougeL': np.float64(0.1305879388043547), 'rougeLsum': np.float64(0.13057145882575127), 'bleu': 0.008060579459331724, 'precisions': [0.1393834315708844, 0.015150828366158239, 0.002885248281470899, 0.0006928422001439497], 'brevity_penalty': 1.0, 'length_ratio': 2.344378583590091, 'translation_length': 629257, 'reference_length': 268411, 'meteor': np.float64(0.20575618616203697)}
{'rouge1': np.float64(0.3697133817563203), 'rouge2': np.float64(0.14545807780193187), 'rougeL': np.float64(0.29275466055978205), 'rougeLsum': np.float64(0.2927958590123075), 'bleu': 0.08575821770005569, 'precisions': [0.40644779678541654, 0.14019510370421412, 0.06969353837162581, 0.037039710389552706], 'brevity_penalty': 0.7787110140888819, 'length_ratio': 0.7999262325314537, 'translation_length': 214709, 'reference_length': 268411, 'meteor': np.float64(0.2937659284912623)}
